In [24]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Computing metrics for each language and feature set, and saving them to CSV files

In [ ]:
def compute_binary_metrics(
    df,
    y_true_col="y_true",
    y_pred_col="y_pred",
    prob_col="prob_class_1",
    positive_class=1
):
    """
    Computes binary classification metrics for one dataframe subset.

    Returns one row as a pandas Series.
    """

    if df.empty:
        return pd.Series({
            "accuracy": np.nan,
            "balanced_accuracy": np.nan,
            "precision": np.nan,
            "recall": np.nan,
            "f1": np.nan,
            "auc": np.nan,
            "n_samples": 0
        })

    y_true = df[y_true_col]
    y_pred = df[y_pred_col]

    accuracy = accuracy_score(y_true, y_pred)
    balanced_accuracy = balanced_accuracy_score(y_true, y_pred)

    precision = precision_score(
        y_true,
        y_pred,
        pos_label=positive_class,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        pos_label=positive_class,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        pos_label=positive_class,
        zero_division=0
    )

    # AUC needs probability 
    if prob_col in df.columns and y_true.nunique() == 2:
        auc = roc_auc_score(y_true, df[prob_col])
    else:
        auc = np.nan

    return pd.Series({
        "accuracy": accuracy,
        "balanced_accuracy": balanced_accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": auc,
        "n_samples": len(df)
    })
def compute_metrics_by_group(
    df,
    group_cols,
    y_true_col="y_true",
    y_pred_col="y_pred",
    prob_col="prob_class_1",
    positive_class=1
):
    """
    Compute binary classification metrics for each group.

    Example:
        compute_metrics_by_group(
            mono_eng,
            group_cols=["feature_set", "model"]
        )
    """

    rows = []

    for group_values, group_df in df.groupby(group_cols):
        metrics = compute_binary_metrics(
            group_df,
            y_true_col=y_true_col,
            y_pred_col=y_pred_col,
            prob_col=prob_col,
            positive_class=positive_class
        )

        if not isinstance(group_values, tuple):
            group_values = (group_values,)

        group_info = dict(zip(group_cols, group_values))
        row = {**group_info, **metrics.to_dict()}
        rows.append(row)

    return pd.DataFrame(rows)

In [ ]:
results_df = pd.read_csv(r"D:\masteruwefduyqeahfdqe\ASR-project\Interpretable features\model_predictions_final_new_cross.csv")
results_df

In [27]:
results_df[['model', 'language', 'feature_set', 'experiment_name']].value_counts()

model         language  feature_set  experiment_name                  
KNN           English   all          train_mandarin_greek_test_english    159
SVM-RBF       English   all          train_mandarin_greek_test_english    159
RandomForest  English   subset_cha   train_mandarin_greek_test_english    159
                        subset_wav   train_mandarin_greek_test_english    159
                        wav_only     train_mandarin_greek_test_english    159
                                                                         ... 
SVM-RBF       Greek     cha_only     train_english_mandarin_test_greek     17
                        subset_cha   train_english_mandarin_test_greek     17
                        subset_wav   train_english_mandarin_test_greek     17
SVM-Linear    Greek     all          train_english_mandarin_test_greek     17
MLP           Greek     subset_cha   train_english_mandarin_test_greek     17
Name: count, Length: 90, dtype: int64

In [ ]:
# mono_eng_all.to_csv("mono_english_all_features_prediction.csv", index=False)

In [ ]:
# iterating over languages and feature sets to compute metrics and save them to CSV files
for language in ["English", "Greek", "Mandarin"]:
    for feature_set in ["all", "cha_only", "wav_only", "subset_wav","subset_cha"]:
        for strategy in ["cross"]:
            subset = results_df[
                (results_df["strategy"] == strategy) &
                (results_df["language"] == language) &
                (results_df["feature_set"] == feature_set)]
            
            print(f"{strategy}_{language.lower()}_{feature_set}_features_metrics_new_cross.csv")
            # display(subset)
            # computing metrics
            metrics = compute_metrics_by_group(subset, group_cols=["model"])
            display(metrics)
            # saving metrics to CSV
            metrics.to_csv(f"{strategy}_{language.lower()}_{feature_set}_features_metrics_new_cross.csv", index=False)